In [1]:
import torch
import pyro
import pyro.distributions as dist
from pyro.infer import SVI, Trace_ELBO, Predictive
from pyro.infer.autoguide import AutoNormal
from pyro.optim import ClippedAdam

pyro.clear_param_store()

torch.set_default_dtype(torch.float32)

In [13]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

train_df = pd.read_parquet("airport_15min_train.parquet")
test_df = pd.read_parquet("airport_15min_test.parquet")

train_df["timestamp"] = pd.to_datetime(train_df["timestamp"], utc=True)
test_df["timestamp"] = pd.to_datetime(test_df["timestamp"], utc=True)

train_df.head()

,timestamp,ins,outs,total,outs_share,slot,weekday,month,is_weekend
0,2024-04-01 00:00:00+00:00,413.0,84.0,497.0,0.169014,0,0,3,0
1,2024-04-01 00:15:00+00:00,182.0,36.0,218.0,0.165138,1,0,3,0
2,2024-04-01 00:30:00+00:00,104.0,55.0,159.0,0.345912,2,0,3,0
3,2024-04-01 00:45:00+00:00,49.0,34.0,83.0,0.409639,3,0,3,0
4,2024-04-01 01:00:00+00:00,28.0,26.0,54.0,0.481481,4,0,3,0


In [3]:
# Preparing tensors

train_df = train_df.sort_values("timestamp").reset_index(drop=True)
test_df = test_df.sort_values("timestamp").reset_index(drop=True)

slot_train = torch.tensor(train_df["slot"].values, dtype=torch.long)
total_train = torch.tensor(train_df["total"].values, dtype=torch.float32)
outs_train = torch.tensor(train_df["outs"].values, dtype=torch.float32)

slot_test = torch.tensor(test_df["slot"].values, dtype=torch.long)
total_test = torch.tensor(test_df["total"].values, dtype=torch.float32)
outs_test = torch.tensor(test_df["outs"].values, dtype=torch.float32)

In [9]:
def airport_model(slot, total_obs=None, outs_obs=None):
    n = len(slot)
    n_slots = 96

    mu = pyro.sample("mu", dist.Normal(5.5, 1.0))
    eta = pyro.sample("eta", dist.Normal(0.0, 1.0))

    sigma_slot_total = pyro.sample("sigma_slot_total", dist.HalfNormal(1.0))
    sigma_slot_dir = pyro.sample("sigma_slot_dir", dist.HalfNormal(1.0))

    phi = pyro.sample("phi", dist.HalfNormal(10.0))

    with pyro.plate("slots", n_slots):
        slot_total_raw = pyro.sample("slot_total_raw", dist.Normal(0.0, 1.0))
        slot_dir_raw = pyro.sample("slot_dir_raw", dist.Normal(0.0, 1.0))

    slot_total_effect = sigma_slot_total * slot_total_raw
    slot_dir_effect = sigma_slot_dir * slot_dir_raw

    log_lambda = mu + slot_total_effect[slot]
    lambda_ = torch.exp(log_lambda)

    logits_p = eta + slot_dir_effect[slot]

    with pyro.plate("data", n):

        total = pyro.sample(
            "total",
            dist.NegativeBinomial(
                total_count=phi,
                logits=torch.log(lambda_) - torch.log(phi)
            ),
            obs=total_obs
        )

        binomial_total = total_obs if total_obs is not None else total

        pyro.sample(
            "outs",
            dist.Binomial(
                total_count=binomial_total,
                logits=logits_p
            ),
            obs=outs_obs
        )

In [10]:
guide = AutoNormal(airport_model)

pyro.clear_param_store()


optimizer = ClippedAdam({"lr": 0.01, "clip_norm": 10.0})
svi = SVI(airport_model, guide, optimizer, loss=Trace_ELBO())

losses = []

num_steps = 3000

for step in range(num_steps):
    loss = svi.step(slot_train, total_train, outs_train)
    losses.append(loss / len(train_df))

    if step % 300 == 0:
        print(f"step {step:04d} | loss per obs = {losses[-1]:.2f}")

step 0000 | loss per obs = 243.75
step 0300 | loss per obs = 13.15
step 0600 | loss per obs = 12.79
step 0900 | loss per obs = 12.79
step 1200 | loss per obs = 12.78
step 1500 | loss per obs = 12.78
step 1800 | loss per obs = 12.78
step 2100 | loss per obs = 12.78
step 2400 | loss per obs = 12.78
step 2700 | loss per obs = 12.78


In [17]:
predictive = Predictive(
    airport_model,
    guide=guide,
    num_samples=500,
    return_sites=("total", "outs")
)

samples = predictive(slot_test, None, None)

total_samples = samples["total"].detach().cpu().numpy()
outs_samples = samples["outs"].detach().cpu().numpy()

total_pred_mean = total_samples.mean(axis=0)
outs_pred_mean = outs_samples.mean(axis=0)

test_df = test_df.copy()

test_df["pyro_total_mean"] = total_pred_mean
test_df["pyro_outs_mean"] = outs_pred_mean
test_df["pyro_ins_mean"] = test_df["pyro_total_mean"] - test_df["pyro_outs_mean"]
test_df["pyro_outs_share_mean"] = (
    test_df["pyro_outs_mean"] / np.maximum(test_df["pyro_total_mean"], 1)
)

test_df[[
    "total",
    "outs",
    "pyro_total_mean",
    "pyro_outs_mean",
    "pyro_ins_mean",
    "pyro_outs_share_mean"
]].head()

,total,outs,pyro_total_mean,pyro_outs_mean,pyro_ins_mean,pyro_outs_share_mean
0,187.0,76.0,267.648010,77.559998,190.088013,0.289784
1,317.0,84.0,239.880005,66.031998,173.848007,0.275271
2,219.0,75.0,211.746002,65.636002,146.110001,0.309975
3,162.0,74.0,187.753998,54.980000,132.774002,0.292830
4,107.0,35.0,185.292007,55.094002,130.197998,0.297336


In [18]:
print(test_df.shape)
print(total_samples.shape)
print(outs_samples.shape)

(13822, 31)
(500, 13822)
(500, 13822)


# Evaluating Pyro Model

In [19]:
def mae(y, yhat):
    return np.mean(np.abs(y - yhat))

def rmse(y, yhat):
    return np.sqrt(np.mean((y - yhat) ** 2))

pyro_metrics = {
    "total_mae": mae(test_df["total"], test_df["pyro_total_mean"]),
    "total_rmse": rmse(test_df["total"], test_df["pyro_total_mean"]),
    "outs_mae": mae(test_df["outs"], test_df["pyro_outs_mean"]),
    "outs_rmse": rmse(test_df["outs"], test_df["pyro_outs_mean"]),
    "ins_mae": mae(test_df["ins"], test_df["pyro_ins_mean"]),
    "ins_rmse": rmse(test_df["ins"], test_df["pyro_ins_mean"]),
    "direction_mae": mae(test_df["outs_share"], test_df["pyro_outs_share_mean"]),
}

true_direction = test_df["outs"] > test_df["ins"]
pred_direction = test_df["pyro_outs_mean"] > test_df["pyro_ins_mean"]

pyro_metrics["direction_accuracy"] = np.mean(true_direction == pred_direction)

pyro_metrics

{'total_mae': np.float64(77.70922652313426),
 'total_rmse': np.float64(100.13134754240224),
 'outs_mae': np.float64(42.31844851429393),
 'outs_rmse': np.float64(56.42212503506528),
 'ins_mae': np.float64(43.428643924181586),
 'ins_rmse': np.float64(58.057135965109026),
 'direction_mae': np.float64(0.08789713674558694),
 'direction_accuracy': np.float64(0.8081319635363913)}

In [20]:
baseline_results = pd.read_csv("baseline_results.csv", index_col=0)

comparison = baseline_results.copy()
comparison.loc["pyro_slot_model"] = pyro_metrics

comparison

,total_mae,total_rmse,outs_mae,outs_rmse,ins_mae,ins_rmse,direction_mae,direction_accuracy
global,131.503793,160.122811,81.755828,96.869366,69.070972,84.139071,0.161458,0.590870
slot,75.831100,97.738296,41.842536,55.594710,42.235025,56.325320,0.086621,0.808060
slot_weekday,64.074949,83.278428,35.838424,48.080332,37.193288,50.036198,0.082968,0.808132
pyro_slot_model,77.709227,100.131348,42.318449,56.422125,43.428644,58.057136,0.087897,0.808132


In [21]:
total_coverage = np.mean(
    (test_df["total"] >= test_df["pyro_total_q05"]) &
    (test_df["total"] <= test_df["pyro_total_q95"])
)

outs_coverage = np.mean(
    (test_df["outs"] >= test_df["pyro_outs_q05"]) &
    (test_df["outs"] <= test_df["pyro_outs_q95"])
)

share_coverage = np.mean(
    (test_df["outs_share"] >= test_df["pyro_outs_share_q05"]) &
    (test_df["outs_share"] <= test_df["pyro_outs_share_q95"])
)

print(f"Total 90% coverage:      {total_coverage:.3f}")
print(f"Outs 90% coverage:       {outs_coverage:.3f}")
print(f"Out-share 90% coverage:  {share_coverage:.3f}")

Total 90% coverage:      0.890
Outs 90% coverage:       0.907
Out-share 90% coverage:  0.441


# Model 2: slot + weekday

In [23]:
weekday_train = torch.tensor(train_df["weekday"].values, dtype=torch.long)
weekday_test = torch.tensor(test_df["weekday"].values, dtype=torch.long)

def airport_model_slot_weekday(slot, weekday, total_obs=None, outs_obs=None):
    n = len(slot)
    n_slots = 96
    n_weekdays = 7

    mu = pyro.sample("mu", dist.Normal(5.5, 1.0))
    eta = pyro.sample("eta", dist.Normal(0.0, 1.0))

    sigma_slot_total = pyro.sample("sigma_slot_total", dist.HalfNormal(1.0))
    sigma_slot_dir = pyro.sample("sigma_slot_dir", dist.HalfNormal(1.0))

    sigma_weekday_total = pyro.sample("sigma_weekday_total", dist.HalfNormal(0.5))
    sigma_weekday_dir = pyro.sample("sigma_weekday_dir", dist.HalfNormal(0.5))

    phi = pyro.sample("phi", dist.HalfNormal(10.0))

    with pyro.plate("slots", n_slots):
        slot_total_raw = pyro.sample("slot_total_raw", dist.Normal(0.0, 1.0))
        slot_dir_raw = pyro.sample("slot_dir_raw", dist.Normal(0.0, 1.0))

    with pyro.plate("weekdays", n_weekdays):
        weekday_total_raw = pyro.sample("weekday_total_raw", dist.Normal(0.0, 1.0))
        weekday_dir_raw = pyro.sample("weekday_dir_raw", dist.Normal(0.0, 1.0))

    slot_total_effect = sigma_slot_total * slot_total_raw
    slot_dir_effect = sigma_slot_dir * slot_dir_raw

    weekday_total_effect = sigma_weekday_total * weekday_total_raw
    weekday_dir_effect = sigma_weekday_dir * weekday_dir_raw

    log_lambda = (
        mu
        + slot_total_effect[slot]
        + weekday_total_effect[weekday]
    )

    lambda_ = torch.exp(log_lambda)

    logits_p = (
        eta
        + slot_dir_effect[slot]
        + weekday_dir_effect[weekday]
    )

    with pyro.plate("data", n):
        total = pyro.sample(
            "total",
            dist.NegativeBinomial(
                total_count=phi,
                logits=torch.log(lambda_) - torch.log(phi)
            ),
            obs=total_obs
        )

        binomial_total = total_obs if total_obs is not None else total

        pyro.sample(
            "outs",
            dist.Binomial(
                total_count=binomial_total,
                logits=logits_p
            ),
            obs=outs_obs
        )

In [24]:
pyro.clear_param_store()

guide_sw = AutoNormal(airport_model_slot_weekday)

optimizer = ClippedAdam({"lr": 0.01, "clip_norm": 10.0})
svi = SVI(airport_model_slot_weekday, guide_sw, optimizer, loss=Trace_ELBO())

losses_sw = []

num_steps = 3000

for step in range(num_steps):
    loss = svi.step(slot_train, weekday_train, total_train, outs_train)
    losses_sw.append(loss / len(train_df))

    if step % 300 == 0:
        print(f"step {step:04d} | loss per obs = {losses_sw[-1]:.2f}")

step 0000 | loss per obs = 248.02
step 0300 | loss per obs = 12.88
step 0600 | loss per obs = 12.73
step 0900 | loss per obs = 12.69
step 1200 | loss per obs = 12.70
step 1500 | loss per obs = 12.69
step 1800 | loss per obs = 12.69
step 2100 | loss per obs = 12.69
step 2400 | loss per obs = 12.68
step 2700 | loss per obs = 12.68


In [25]:
predictive_sw = Predictive(
    airport_model_slot_weekday,
    guide=guide_sw,
    num_samples=500,
    return_sites=("total", "outs")
)

samples_sw = predictive_sw(slot_test, weekday_test, None, None)

total_samples_sw = samples_sw["total"].detach().cpu().numpy()
outs_samples_sw = samples_sw["outs"].detach().cpu().numpy()

test_df["pyro_sw_total_mean"] = total_samples_sw.mean(axis=0)
test_df["pyro_sw_outs_mean"] = outs_samples_sw.mean(axis=0)
test_df["pyro_sw_ins_mean"] = test_df["pyro_sw_total_mean"] - test_df["pyro_sw_outs_mean"]
test_df["pyro_sw_outs_share_mean"] = (
    test_df["pyro_sw_outs_mean"] / np.maximum(test_df["pyro_sw_total_mean"], 1)
)

test_df["pyro_sw_total_q05"] = np.quantile(total_samples_sw, 0.05, axis=0)
test_df["pyro_sw_total_q95"] = np.quantile(total_samples_sw, 0.95, axis=0)

test_df["pyro_sw_outs_q05"] = np.quantile(outs_samples_sw, 0.05, axis=0)
test_df["pyro_sw_outs_q95"] = np.quantile(outs_samples_sw, 0.95, axis=0)

outs_share_samples_sw = outs_samples_sw / np.maximum(total_samples_sw, 1)

test_df["pyro_sw_outs_share_q05"] = np.quantile(outs_share_samples_sw, 0.05, axis=0)
test_df["pyro_sw_outs_share_q95"] = np.quantile(outs_share_samples_sw, 0.95, axis=0)

In [26]:
pyro_sw_metrics = {
    "total_mae": mae(test_df["total"], test_df["pyro_sw_total_mean"]),
    "total_rmse": rmse(test_df["total"], test_df["pyro_sw_total_mean"]),
    "outs_mae": mae(test_df["outs"], test_df["pyro_sw_outs_mean"]),
    "outs_rmse": rmse(test_df["outs"], test_df["pyro_sw_outs_mean"]),
    "ins_mae": mae(test_df["ins"], test_df["pyro_sw_ins_mean"]),
    "ins_rmse": rmse(test_df["ins"], test_df["pyro_sw_ins_mean"]),
    "direction_mae": mae(test_df["outs_share"], test_df["pyro_sw_outs_share_mean"]),
}

true_direction = test_df["outs"] > test_df["ins"]
pred_direction_sw = test_df["pyro_sw_outs_mean"] > test_df["pyro_sw_ins_mean"]

pyro_sw_metrics["direction_accuracy"] = np.mean(true_direction == pred_direction_sw)

comparison.loc["pyro_slot_weekday_model"] = pyro_sw_metrics
comparison

,total_mae,total_rmse,outs_mae,outs_rmse,ins_mae,ins_rmse,direction_mae,direction_accuracy
global,131.503793,160.122811,81.755828,96.869366,69.070972,84.139071,0.161458,0.590870
slot,75.831100,97.738296,41.842536,55.594710,42.235025,56.325320,0.086621,0.808060
slot_weekday,64.074949,83.278428,35.838424,48.080332,37.193288,50.036198,0.082968,0.808132
pyro_slot_model,77.709227,100.131348,42.318449,56.422125,43.428644,58.057136,0.087897,0.808132
pyro_slot_weekday_model,74.948653,96.662988,41.197572,55.021653,42.072216,55.903204,0.087823,0.808566


In [27]:
total_coverage_sw = np.mean(
    (test_df["total"] >= test_df["pyro_sw_total_q05"]) &
    (test_df["total"] <= test_df["pyro_sw_total_q95"])
)

outs_coverage_sw = np.mean(
    (test_df["outs"] >= test_df["pyro_sw_outs_q05"]) &
    (test_df["outs"] <= test_df["pyro_sw_outs_q95"])
)

share_coverage_sw = np.mean(
    (test_df["outs_share"] >= test_df["pyro_sw_outs_share_q05"]) &
    (test_df["outs_share"] <= test_df["pyro_sw_outs_share_q95"])
)

print(f"Total 90% coverage:      {total_coverage_sw:.3f}")
print(f"Outs 90% coverage:       {outs_coverage_sw:.3f}")
print(f"Out-share 90% coverage:  {share_coverage_sw:.3f}")

Total 90% coverage:      0.886
Outs 90% coverage:       0.907
Out-share 90% coverage:  0.432
